# Cross-session recovery test of the Stokes MeshGraphNet training (Kaggle GPU)

Launcher for `scripts/recovery_check.py` of [physicsnemo-stokes-finetuning](https://github.com/AdebanjiAdelowo/physicsnemo-stokes-finetuning). It checks out one commit and calls its scripts; it contains no model or training code. Session settings: **Accelerator** an NVIDIA GPU, **Internet** on.

Two separate executions:

| `STAGE` | What runs | Output |
|---|---|---|
| `"first"` | set-up and tests; `recovery_check.py first`: the upstream architecture on 50 training meshes, paused after epoch 3 of 6 (`recovery_resumed`), and the same run without interruption (`recovery_reference`); both run directories packed | `state-recovery_resumed.tar`, `state-recovery_reference.tar` (with `.sha256` and `.json`) |
| `"second"` | set-up and tests; the two tar files restored from the first session's output (attached as a kernel source) and verified; `recovery_check.py second`: corruption and incompatibility drills, resume to epoch 6, an uninterrupted repeat, comparison against the reference | `recovery_check.json` |

`DETERMINISTIC = True` runs both sessions with deterministic CUDA algorithms and decides by bit-identity (criterion version 3). The pass criteria are in `src/stokes_ft/recovery_check.py`. The numbers verify continuation; they are not results.

In [ ]:
REPO_URL = "https://github.com/AdebanjiAdelowo/physicsnemo-stokes-finetuning.git"
STAGE = "first"                               # "first" or "second"; the second session needs the first one's output as a kernel source
DETERMINISTIC = True                          # deterministic CUDA algorithms in both sessions: bit-identity criterion (v3)
REF = "e0c57f7a1e7643e994a89ac6036972ff0f4f5daa"   # output of `git rev-parse HEAD` for the commit to run
SESSION_BUDGET_HOURS = 9.0                    # GPU time you are willing to spend in this session; check Kaggle's current limits
GITHUB_TOKEN_SECRET = None                    # name of a Kaggle secret holding a GitHub token; only for a private repository

import re, time
T0 = time.time()
assert re.fullmatch(r"[0-9a-f]{40}", REF), "REF must be a full 40-character commit SHA, not a branch or tag."
WORK = "/kaggle/working"
REPO = f"{WORK}/physicsnemo-stokes-finetuning"

def hours_left():
    return SESSION_BUDGET_HOURS - (time.time() - T0) / 3600

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "CUDA is not available. Select a GPU accelerator; this notebook does not fall back to CPU."
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
import os, subprocess
def sh(cmd, cwd=None):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

def tree_is_clean():
    return not subprocess.run(["git", "status", "--porcelain"], cwd=REPO, capture_output=True, text=True).stdout.strip()

if not os.path.isdir(REPO):
    url = REPO_URL
    if GITHUB_TOKEN_SECRET:
        from kaggle_secrets import UserSecretsClient
        url = REPO_URL.replace("https://", "https://" + UserSecretsClient().get_secret(GITHUB_TOKEN_SECRET) + "@")
    subprocess.run(["git", "clone", "--quiet", url, REPO], check=True)   # not echoed: the URL may hold a token
    subprocess.run(["git", "remote", "set-url", "origin", REPO_URL], cwd=REPO, check=True)
assert tree_is_clean(), "the existing working tree is not clean"
subprocess.run(["git", "checkout", "--quiet", REF], cwd=REPO, check=True)
head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True).stdout.strip()
assert head == REF, f"checked out {head}, expected {REF}"
assert tree_is_clean(), "the working tree is not clean"
print("commit", head)

In [ ]:
sh("pip install -q -r requirements.txt", cwd=REPO)
sh("FORCE_ONLY_CPU=1 pip install -q --no-build-isolation --no-binary torch-scatter torch-scatter==2.1.2", cwd=REPO)
# The Kaggle image preinstalls RAPIDS (cuml, cudf, cupy, rmm, ...) and numba. PhysicsNeMo imports cuml
# whenever it is installed; cudf then imports numba, which rejects the NumPy that requirements.txt
# installs (first launch, 2026-11-07: "Numba needs NumPy 2.2 or less. Got NumPy 2.5"). The validated
# environment (requirements.lock.txt) contains none of these packages, so they are removed from this
# session instead of downgrading NumPy.
RAPIDS_IMPORTS = ("cuml", "cudf", "dask_cudf", "cugraph", "pylibcugraph", "pylibcugraphops", "nx_cugraph", "cuspatial",
                  "cuvs", "cuxfilter", "cucim", "rmm", "pylibraft", "raft_dask", "cupy", "numba", "numba_cuda")
sh("python -c \"from importlib.metadata import packages_distributions as p; m = p(); "
   f"print(' '.join(sorted({{d for name in {RAPIDS_IMPORTS!r} for d in m.get(name, [])}})))\" > /tmp/rapids_dists.txt")
dists = open("/tmp/rapids_dists.txt").read().split()
print("removing from this session:", dists)
if dists:
    sh("pip uninstall -y -q " + " ".join(dists))
sh('python -c "import torch, physicsnemo, torch_geometric, torch_scatter; assert torch.cuda.is_available(); '
   "print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| physicsnemo', physicsnemo.__version__, "
   "'| torch_geometric', torch_geometric.__version__, '| torch_scatter', torch_scatter.__version__, '|', torch.cuda.get_device_name(0))\"", cwd=REPO)
# every project module imports on this environment before any data or GPU time is spent
sh('PYTHONPATH=src python -c "import numpy, stokes_ft.data, stokes_ft.engine, stokes_ft.model, stokes_ft.study, '
   "stokes_ft.physics, stokes_ft.weak, stokes_ft.variants, stokes_ft.fem; print('project imports OK | numpy', numpy.__version__)\"", cwd=REPO)

In [ ]:
sh("python scripts/prepare_data.py", cwd=REPO)      # 172 MB from NGC, SHA-256 verified, 800/100/100 split with seed 0
assert tree_is_clean(), "preparing the data modified the clone"
sh("python -m pytest -q", cwd=REPO)

## Recovery stage

In [ ]:
import glob
names = ("resumed", "reference")
FLAG = " --deterministic" if DETERMINISTIC else ""
if STAGE == "first":
    sh("python scripts/recovery_check.py first" + FLAG, cwd=REPO)
    for name in names:
        sh(f"python scripts/session_state.py pack runs/recovery_{name} {WORK}/state-recovery_{name}.tar", cwd=REPO)
    print("RECOVERY TEST, FIRST SESSION COMPLETE")
elif STAGE == "second":
    for name in names:
        found = sorted(glob.glob(f"/kaggle/input/**/state-recovery_{name}.tar", recursive=True))
        assert len(found) == 1, f"expected one state-recovery_{name}.tar under /kaggle/input, found {found}"
        sh(f"python scripts/session_state.py unpack {found[0]} runs", cwd=REPO)
    assert tree_is_clean(), "restoring the state modified the clone"
    try:
        sh("python scripts/recovery_check.py second" + FLAG, cwd=REPO)
    finally:
        if os.path.exists(f"{REPO}/runs/recovery_check.json"):
            sh(f"cp runs/recovery_check.json {WORK}/recovery_check.json", cwd=REPO)
    print("RECOVERY TEST, SECOND SESSION COMPLETE")
else:
    raise ValueError(f"STAGE must be 'first' or 'second', not {STAGE!r}")
